# 06 — TabNet (PyTorch)

TabNet (Arik & Pfister, 2019) uses sequential attentive steps with sparse feature selection
instead of a fixed architecture. Each step learns a soft mask over the input features,
focusing on a different subset at every pass.

| Component | TabNet |
|---|---|
| Architecture | Sequential attentive steps (n_steps varies per feature set) |
| Feature selection | Sparse attention mask per step (Sparsemax) |
| Normalisation | Ghost Batch Normalisation (momentum=0.02) |
| Activation | Gated Linear Unit (GLU) |
| Regularisation | Attention dropout + final dropout + sparsity loss (all vary per feature set) |
| Optimiser | AdamW (weight_decay varies per feature set) |
| LR schedule | CosineAnnealingWarmRestarts (T_0=75, η_min=1e-6) |
| Loss | BCEWithLogitsLoss + pos_weight + λ · sparsity |
| Interpretability | Per-step feature importance maps |
| Patience | Varies per feature set |
| Max epochs | 300 |

Results are appended to `outputs/results/metrics.csv`.  
Plots are saved to `outputs/figures/models/`.

## 0 · Imports

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

'''Self Modules'''
from src.data_loader import load_data
from src.preprocessing import preprocess
from src.models import split_data, get_X_y
from src.tabnet import train_tabnet, TabNetWrapper
from src.evaluation import evaluate_model, plot_confusion_matrix, plot_roc_curve
from config import FEATURE_SETS, FEATURE_CONFIG, FIGURES_MODELS, PLOT_DPI, TABNET_PARAMS, TABNET_DEFAULTS

## 1 · Load & Split

Split is done on the **raw** DataFrame before preprocessing.  
This prevents any data leakage from imputation or encoding statistics.

In [ ]:
df_raw = load_data()

df_train_raw, df_test_raw = split_data(df_raw, test_seasons=[2022, 2023])

## 2 · TabNet Feature Set Loop

Train the TabNet on all feature sets and collect metrics.  
Training curve, confusion matrix, and ROC curve are saved for every feature set.

In [ ]:
# preprocess the largest feature set once; reuse column subsets to avoid redundant work.
LARGEST_FS = max(FEATURE_SETS, key=lambda k: len(FEATURE_SETS[k]))
_df_train_full = preprocess(df_train_raw, feature_set=FEATURE_SETS[LARGEST_FS], feature_config=FEATURE_CONFIG)
_df_test_full  = preprocess(df_test_raw,  feature_set=FEATURE_SETS[LARGEST_FS], feature_config=FEATURE_CONFIG)

# test and train alignment
_df_test_full = _df_test_full.reindex(columns=_df_train_full.columns, fill_value=0)

In [ ]:
# updated feature sets with encoded column names before loop
FEATURE_SETS_ENCODED = {
    fs_name: [
        c for c in _df_train_full.columns
        if c != "target" and any(c == f or c.startswith(f + "_") for f in fs_cols)
    ]
    for fs_name, fs_cols in FEATURE_SETS.items()
}
FEATURE_SETS_ENCODED

In [ ]:
all_metrics = []

for fs_name, fs_cols in FEATURE_SETS_ENCODED.items():

    print(f"\n{'='*55}")
    print(f" TabNet | Feature set: {fs_name}  ({len(fs_cols)} features)")
    print(f"{'='*55}")

    # subset preprocessed data to the current feature set
    df_train = _df_train_full[fs_cols + ["target"]]
    df_test  = _df_test_full[fs_cols + ["target"]]

    X_train, y_train = get_X_y(df_train)
    X_test,  y_test  = get_X_y(df_test)

    # train
    model, history, scaler = train_tabnet(X_train, y_train, **{**TABNET_DEFAULTS, **TABNET_PARAMS[fs_name]})

    wrapper = TabNetWrapper(model, scaler)

    # evaluate
    metrics = evaluate_model(wrapper, X_test, y_test, "TabNet", fs_name)
    all_metrics.append(metrics)

    # --- training curve ---
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history["train_loss"], label="Train loss")
    ax.plot(history["val_loss"],   label="Val loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(f"TabNet training curve — {fs_name}")
    ax.legend()
    fig.tight_layout()
    #fig.savefig(FIGURES_MODELS / f"tabnet_training_curve_{fs_name}.png", dpi=PLOT_DPI)
    plt.show()

    # --- confusion matrix ---
    #fig_cm = plot_confusion_matrix(wrapper, X_test, y_test, "TabNet", fs_name)
    #fig_cm.savefig(FIGURES_MODELS / f"tabnet_confusion_matrix_{fs_name}.png", dpi=PLOT_DPI)
    #plt.show()

    # --- ROC curve ---
    #fig_roc = plot_roc_curve(wrapper, X_test, y_test, "TabNet", fs_name)
    #fig_roc.savefig(FIGURES_MODELS / f"tabnet_roc_curve_{fs_name}.png", dpi=PLOT_DPI)
    #plt.show()

print("\n=== Summary ===")
for m in all_metrics:
    print(m)